## Brain Map Visualization - Cortical + Subcortical Volume (Desikan-Aseg)

### Overview

This notebook visualizes parcelwise **effect sizes (Hedges' g)** of cortical and subcortical
volume differences using the `ggseg` R package (Mowinckel & Vidal-Piñeiro, 2020).

**Atlases:**
- Cortical parcels → Desikan-Killiany `dk()`: 68 parcels, join key `lh_bankssts` format
- Subcortical parcels → FreeSurfer `aseg()`: 16 parcels, join key `Left-Hippocampus` format

**Covariate note:** Volume measures were residualized on **age, SEX, and eTIV**.

**Contrasts:** De Novo PD vs HC · Prodromal PD vs HC · De Novo PD vs Prodromal PD

**Reference:** Mowinckel & Vidal-Piñeiro (2020). *Advances in Methods and Practices in
Psychological Science*.

In [ ]:
# install.packages(c("tidyverse", "patchwork", "remotes"))
# remotes::install_github("LCBC-UiO/ggseg")
suppressPackageStartupMessages({
  library(ggseg); library(tidyverse); library(patchwork)
})
cat("ggseg:", as.character(packageVersion("ggseg")), "\n")

### 1. Load pre-computed parcelwise OLS T-statistics

Six CSV files from the main analysis notebooks, two sets of three contrasts:
- **Cortical** (`volume_cortical_shi.ipynb`): 68 parcels (dk atlas), covariates: age, sex, eTIV
- **Subcortical** (`volume_subcortical_shi.ipynb`): 14 parcels (aseg atlas), covariates: age, sex, eTIV

Each file has columns: `parcel, Tvalue, pvalue, df, hemi`.

**Why pre-computed T-stats?**
Volumetric measures scale with eTIV. Running Welch's t-test on unadjusted volumes can produce
biased or direction-reversed effect sizes, especially for subcortical structures where PD–HC
eTIV differences are large. The OLS T-statistics already account for eTIV (and age, sex),
giving the correct covariate-adjusted result consistent with the main analysis.

In [ ]:
DATA_DIR    <- "../../data"
RESULTS_DIR <- "../../results"
FIG_DIR     <- file.path(RESULTS_DIR, "figures", "ggseg_volume_desikanaseg")
dir.create(FIG_DIR, recursive = TRUE, showWarnings = FALSE)

GROUP_N <- list(
  "De Novo PD vs HC"           = c(n1 = 558, n2 = 192),
  "Prodromal PD vs HC"         = c(n1 = 279, n2 = 192),
  "De Novo PD vs Prodromal PD" = c(n1 = 558, n2 = 279)
)

CORTICAL_FILES <- list(
  "De Novo PD vs HC"           = file.path(RESULTS_DIR, "de novo pd vs hc_parcelwise_ttest_volume_cortical.csv"),
  "Prodromal PD vs HC"         = file.path(RESULTS_DIR, "prodromal pd vs hc_parcelwise_ttest_volume_cortical.csv"),
  "De Novo PD vs Prodromal PD" = file.path(RESULTS_DIR, "de novo pd vs prodromal pd_parcelwise_ttest_volume_cortical.csv")
)

SUBCORTICAL_FILES <- list(
  "De Novo PD vs HC"           = file.path(RESULTS_DIR, "de_novo_pd_vs_hc_parcelwise_ttest_volume_subcortical.csv"),
  "Prodromal PD vs HC"         = file.path(RESULTS_DIR, "prodromal_pd_vs_hc_parcelwise_ttest_volume_subcortical.csv"),
  "De Novo PD vs Prodromal PD" = file.path(RESULTS_DIR, "de_novo_pd_vs_prodromal_pd_parcelwise_ttest_volume_subcortical.csv")
)

# Subcortical structure names (used in cell-labels to detect atlas type)
SUBCORTICAL <- c("Thalamus", "Caudate", "Putamen", "Pallidum",
                 "Hippocampus", "Amygdala", "Accumbens+area")

df_cortical <- imap_dfr(CORTICAL_FILES, function(path, cname) {
  read_csv(path, show_col_types = FALSE) %>% mutate(contrast = cname)
}) %>% filter(!is.na(pvalue))

df_subcortical <- imap_dfr(SUBCORTICAL_FILES, function(path, cname) {
  read_csv(path, show_col_types = FALSE) %>% mutate(contrast = cname)
}) %>% filter(!is.na(pvalue))

df_all <- bind_rows(df_cortical, df_subcortical)
cat("Cortical rows:   ", nrow(df_cortical), "(", nrow(df_cortical)/3, "parcels ×", n_distinct(df_cortical$contrast), "contrasts)\n")
cat("Subcortical rows:", nrow(df_subcortical), "(", nrow(df_subcortical)/3, "parcels ×", n_distinct(df_subcortical$contrast), "contrasts)\n")
cat("Total rows:      ", nrow(df_all), "\n")

### 2. Derive Hedges' g from OLS T-statistics

Convert each OLS T-statistic to Hedges' g:

$$g = -T \cdot \sqrt{\frac{1}{n_1} + \frac{1}{n_2}} \cdot J(df), \quad J(df) = 1 - \frac{3}{4 \cdot df - 1}$$

**Sign convention:** OLS T encodes (reference − first-named), so −T gives first-named − reference.
g > 0 means the first-named group has *greater* volume.

The formula is applied identically to cortical and subcortical parcels — the same contrast
groups (n1, n2) and the same df are used for both, since all parcels come from the same
OLS models run on overlapping samples.

In [ ]:
# Named list kept for downstream visualization cells that loop over names(contrasts)
contrasts <- setNames(as.list(names(GROUP_N)), names(GROUP_N))

hedges_g_from_t <- function(t, n1, n2, df_resid) {
  -t * sqrt(1/n1 + 1/n2) * (1 - 3 / (4 * df_resid - 1))
}

df_stats <- df_all %>%
  rowwise() %>%
  mutate(g = hedges_g_from_t(
    Tvalue,
    GROUP_N[[contrast]][["n1"]],
    GROUP_N[[contrast]][["n2"]],
    df
  )) %>%
  ungroup()

cat("Computed:", nrow(df_stats), "rows\n")

### 3. Build ggseg label columns

ggseg 2.x joins user data to the atlas on a single `label` column:
- **dk atlas** (cortical): `"lh_bankssts"`, `"rh_fusiform"`, …
- **aseg atlas** (subcortical): `"Left-Hippocampus"`, `"Right-Thalamus"`, …

The only non-trivial mapping: `Accumbens+area` → `Accumbens-area` (replace `+` with `-`).

**Important:** only `label` + the fill variable are passed to `ggplot()`. Any extra column
sharing a name with an atlas column (`hemi`, `region`, …) breaks the join.

In [ ]:
df_stats <- df_stats %>%
  mutate(
    raw_label = str_replace(parcel, ".*_lab-", ""),
    hemi_long = if_else(str_detect(parcel, "hemi-L"), "left", "right"),
    hemi_lh   = if_else(hemi_long == "left", "lh",   "rh"),
    hemi_cap  = if_else(hemi_long == "left", "Left",  "Right"),
    type      = if_else(raw_label %in% SUBCORTICAL, "subcortical", "cortical"),
    # ggseg join label
    label = case_when(
      type == "cortical"    ~ paste0(hemi_lh, "_", raw_label),
      type == "subcortical" ~ paste0(hemi_cap, "-", str_replace(raw_label, "\\+", "-"))
    ),
    # Readable columns for tables
    hemi   = hemi_long,
    region = raw_label
  )

cat("Cortical:   ", sum(df_stats$type=="cortical")/n_distinct(df_stats$contrast),  "\n")
cat("Subcortical:", sum(df_stats$type=="subcortical")/n_distinct(df_stats$contrast), "\n")

# Verify subcortical labels against atlas
aseg_labels <- as.data.frame(aseg())$label
sc_bad <- df_stats %>% filter(type=="subcortical", !label %in% aseg_labels) %>% pull(label) %>% unique()
if (length(sc_bad)==0) cat("All subcortical labels match aseg atlas.\n") else cat("Unmatched:", sc_bad, "\n")

### 4. FDR correction

Benjamini-Hochberg FDR applied within each contrast across all 84 parcels jointly.

In [ ]:
df_stats <- df_stats %>%
  group_by(contrast) %>%
  mutate(p_fdr = p.adjust(pvalue, method="fdr")) %>%
  ungroup()

df_stats %>%
  group_by(contrast, type) %>%
  summarise(n_sig_fdr=sum(p_fdr<0.05,na.rm=TRUE), n_sig_unc=sum(pvalue<0.05,na.rm=TRUE),
            n=n(), g_max=round(max(abs(g),na.rm=TRUE),3), .groups="drop")

### 5. Colour scale

Shared diverging scale for both cortical and subcortical maps.
Limits ±0.5 for cortical; ±0.8 for subcortical (effects can be larger in basal ganglia).

In [ ]:
make_scale <- function(limit, name="Hedges' g") {
  scale_fill_gradient2(
    low="#2166AC", mid="white", high="#D6604D",
    midpoint=0, limits=c(-limit, limit),
    oob=scales::squish, name=name, na.value="grey85"
  )
}

scale_cort <- make_scale(0.5)
scale_subc <- make_scale(0.8)

### 6. Cortical maps (dk atlas)

68 cortical parcels coloured by Hedges' g.
Left panel: all parcels. Right panel: FDR-masked (q < 0.05).

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}

for (cname in names(contrasts)) {
  d <- df_stats %>% filter(contrast == cname, type == "cortical")

  p_full <- ggplot(d %>% select(label, g)) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g), colour = "white",
               position = position_brain("horizontal")) +
    scale_cort +
    labs(title = cname, subtitle = "Cortical volume - Hedges' g (all parcels; eTIV-corrected)") +
    theme_brain2() +
    theme(plot.title = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  p_mask <- ggplot(d %>% mutate(g_sig = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_sig)) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_sig), colour = "white",
               position = position_brain("horizontal")) +
    scale_cort +
    labs(title = cname, subtitle = "Cortical volume - FDR-masked (q < 0.05 across 82 parcels)") +
    theme_brain2() +
    theme(plot.title = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  combined <- p_full + p_mask + plot_layout(guides = "collect") & theme(legend.position = "right")
  print(combined)
  fname <- paste0("cortical_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(file.path(FIG_DIR, paste0(fname, ".png")), combined, width = 14, height = 5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 7. Subcortical maps (aseg atlas)

16 subcortical structures coloured by Hedges' g.
The aseg atlas join key format is `"Left-Hippocampus"`, `"Right-Thalamus"`, etc.

**PD context:** striatal (caudate, putamen) dopaminergic denervation; hippocampal/amygdalar
atrophy in later stages or cognitive impairment.

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 3.5)

if (!exists("coronal_join")) {
  coronal_join <- function(df) {
    brain_join(df, aseg()) %>% dplyr::filter(view == "coronal_1")
  }
}

for (cname in names(contrasts)) {
  d <- df_stats %>% filter(contrast == cname, type == "subcortical")

  p_full <- ggplot(coronal_join(d %>% select(label, g))) +
    geom_sf(aes(fill = g), colour = "white") +
    scale_subc +
    labs(title = cname, subtitle = "Subcortical volume - Hedges' g (all structures; eTIV-corrected)") +
    theme_void() +
    theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  p_mask <- ggplot(coronal_join(d %>% mutate(g_sig = if_else(p_fdr < 0.05, g, NA_real_)) %>% select(label, g_sig))) +
    geom_sf(aes(fill = g_sig), colour = "white") +
    scale_subc +
    labs(title = cname, subtitle = "Subcortical volume - FDR-masked (q < 0.05; eTIV-corrected)") +
    theme_void() +
    theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  combined <- p_full + p_mask + plot_layout(guides = "collect") & theme(legend.position = "right")
  print(combined)
  fname <- paste0("subcortical_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(file.path(FIG_DIR, paste0(fname, ".png")), combined, width = 14, height = 3.5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 7b. Paper-style brain maps (sequential red scale, FDR-only)

Replicating the style of Laansma et al. (following the figure caption):
> *"Cohen's d values were calculated and are presented in the figure when the FDR-adjusted
> p value reached < 0.05. Darker red indicates more atrophy."*

**Design choices:**
- **Sequential white → dark red** scale: encodes *magnitude* of atrophy, not direction
- **Absolute |Hedges' g|** displayed — only for FDR-significant parcels
- **Non-significant parcels** → very light grey (`#f5f5f5`) so the brain outline stays visible
- **Grey parcel borders** (`grey60`) make individual parcels distinguishable
- One panel per contrast for both cortical (dk) and subcortical (aseg)

In [ ]:
FIG_DIR_PAPER <- file.path(FIG_DIR, "paper_style")
dir.create(FIG_DIR_PAPER, recursive = TRUE, showWarnings = FALSE)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}
if (!exists("coronal_join")) {
  coronal_join <- function(df) {
    brain_join(df, aseg()) %>% dplyr::filter(view == "coronal_1")
  }
}

g_max_cort <- ceiling(max(abs(df_stats$g[df_stats$type == "cortical"]),    na.rm = TRUE) * 10) / 10
g_max_subc <- ceiling(max(abs(df_stats$g[df_stats$type == "subcortical"]), na.rm = TRUE) * 10) / 10
cat("Cortical limit:", g_max_cort, "  Subcortical limit:", g_max_subc, "\n")

scale_cort_paper <- scale_fill_gradient(
  low = "white", high = "#B2182B",
  limits = c(0, g_max_cort), oob = scales::squish,
  name = "|Hedges' g|", na.value = "#f5f5f5"
)
scale_subc_paper <- scale_fill_gradient(
  low = "white", high = "#B2182B",
  limits = c(0, g_max_subc), oob = scales::squish,
  name = "|Hedges' g|", na.value = "#f5f5f5"
)

# ── Cortical paper maps ────────────────────────────────────────────────────────
cat("=== Cortical (dk atlas, 4 views) ===\n")
options(repr.plot.width = 10, repr.plot.height = 4)
for (cname in names(contrasts)) {
  plot_data <- df_stats %>%
    filter(contrast == cname, type == "cortical") %>%
    mutate(g_abs = if_else(p_fdr < 0.05, abs(g), NA_real_)) %>%
    select(label, g_abs)

  p <- ggplot(plot_data) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_abs),
               colour = "grey60", position = position_brain("horizontal")) +
    scale_cort_paper +
    labs(title = cname, subtitle = "Cortical volume - |Hedges' g| (FDR q < 0.05; eTIV-corrected)") +
    theme_brain2() +
    theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  print(p)
  fname <- paste0("paper_cortical_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(file.path(FIG_DIR_PAPER, paste0(fname, ".png")), p, width = 10, height = 4, dpi = 300)
  cat("Saved:", fname, "\n")
}

# ── Subcortical paper maps (coronal slice) ─────────────────────────────────────
cat("\n=== Subcortical (aseg atlas, coronal slice) ===\n")
options(repr.plot.width = 10, repr.plot.height = 3.5)
for (cname in names(contrasts)) {
  plot_data <- df_stats %>%
    filter(contrast == cname, type == "subcortical") %>%
    mutate(g_abs = if_else(p_fdr < 0.05, abs(g), NA_real_)) %>%
    select(label, g_abs)

  p <- ggplot(coronal_join(plot_data)) +
    geom_sf(aes(fill = g_abs), colour = "grey60") +
    scale_subc_paper +
    labs(title = cname, subtitle = "Subcortical volume - |Hedges' g| (FDR q < 0.05; eTIV-corrected)") +
    theme_void() +
    theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5, size = 10, colour = "grey40"),
          legend.position = "right")

  print(p)
  fname <- paste0("paper_subcortical_", tolower(str_replace_all(cname, " ", "_")))
  ggsave(file.path(FIG_DIR_PAPER, paste0(fname, ".png")), p, width = 10, height = 3.5, dpi = 300)
  cat("Saved:", fname, "\n")
}

### 7c. Combined DesikanAseg atrophy maps: cortical 4-view + subcortical coronal, white → pink

Two stacked combined figures (one per threshold level), each with 3 panels (a/b/c, one per contrast):
- **Top row of each panel** — cortical volume reduction (4 views: LH lateral | LH medial | RH medial | RH lateral)
- **Bottom row of each panel** — subcortical volume reduction (coronal MRI slice)
- Pink intensity encodes |Hedges' g| (atrophy magnitude); grey = no reduction or greater volume in reference group

**Figure 7c-1** — Unthresholded: all parcels with g < 0 shown in pink  
**Figure 7c-2** — FDR-corrected: only parcels with q < 0.05 (jointly across 82 structures) shown in pink

In [ ]:
# ── Ensure helpers exist ───────────────────────────────────────────────────────
if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}
if (!exists("coronal_join")) {
  coronal_join <- function(df) {
    brain_join(df, aseg()) %>% dplyr::filter(view == "coronal_1")
  }
}

# ── Unified atrophy scale ──────────────────────────────────────────────────────
max_g_all <- ceiling(max(abs(df_stats$g[df_stats$g < 0]), na.rm = TRUE) * 20) / 20

scale_atrophy_dka <- scale_fill_gradient(
  low = "white", high = "#C0392B",
  limits = c(0, max_g_all), oob = scales::squish,
  name = "Hedges' g\n(atrophy)", na.value = "grey90"
)

# ── Per-contrast descriptions ──────────────────────────────────────────────────
contrast_desc_dka_cort <- c(
  "De Novo PD vs HC"           = "De Novo PD shows less cortical volume than HC",
  "Prodromal PD vs HC"         = "Prodromal PD shows less cortical volume than HC",
  "De Novo PD vs Prodromal PD" = "De Novo PD shows less cortical volume than Prodromal PD"
)
contrast_desc_dka_subc <- c(
  "De Novo PD vs HC"           = "De Novo PD shows less subcortical volume than HC",
  "Prodromal PD vs HC"         = "Prodromal PD shows less subcortical volume than HC",
  "De Novo PD vs Prodromal PD" = "De Novo PD shows less subcortical volume than Prodromal PD"
)

# ── Panel builder ──────────────────────────────────────────────────────────────
make_dka_panel <- function(cname, fdr_only = FALSE) {
  d_cort <- df_stats %>% filter(contrast == cname, type == "cortical")
  d_subc <- df_stats %>% filter(contrast == cname, type == "subcortical")

  n_unc_cort <- sum(d_cort$pvalue < 0.05, na.rm = TRUE)
  n_unc_subc <- sum(d_subc$pvalue < 0.05, na.rm = TRUE)
  n_fdr_cort <- sum(d_cort$p_fdr  < 0.05, na.rm = TRUE)
  n_fdr_subc <- sum(d_subc$p_fdr  < 0.05, na.rm = TRUE)

  if (fdr_only) {
    d_cort <- d_cort %>% mutate(atrophy = if_else(p_fdr < 0.05 & g < 0, abs(g), NA_real_))
    d_subc <- d_subc %>% mutate(atrophy = if_else(p_fdr < 0.05 & g < 0, abs(g), NA_real_))
    info_cort <- paste0(contrast_desc_dka_cort[[cname]], " | ", n_fdr_cort, " parcels FDR q<0.05")
    info_subc <- paste0(contrast_desc_dka_subc[[cname]], " | ", n_fdr_subc, " structures FDR q<0.05")
  } else {
    d_cort <- d_cort %>% mutate(atrophy = if_else(g < 0, abs(g), NA_real_))
    d_subc <- d_subc %>% mutate(atrophy = if_else(g < 0, abs(g), NA_real_))
    info_cort <- paste0(contrast_desc_dka_cort[[cname]], " | ", n_unc_cort, " parcels p<0.05 uncorrected")
    info_subc <- paste0(contrast_desc_dka_subc[[cname]], " | ", n_unc_subc, " structures p<0.05 uncorrected")
  }

  p_cort <- ggplot(d_cort %>% select(label, atrophy)) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = atrophy), colour = "grey70",
               position = position_brain("horizontal")) +
    scale_atrophy_dka +
    labs(subtitle = info_cort) +
    theme_brain2() +
    theme(
      plot.subtitle   = element_text(hjust = 0.5, size = 8.5, colour = "grey40",
                                     margin = margin(b = 4)),
      legend.position = "none",
      plot.margin     = margin(t = 4, r = 2, b = 2, l = 2)
    )

  p_subc <- ggplot(coronal_join(d_subc %>% select(label, atrophy))) +
    geom_sf(aes(fill = atrophy), colour = "grey70") +
    scale_atrophy_dka +
    labs(subtitle = info_subc) +
    theme_void() +
    theme(
      plot.subtitle   = element_text(hjust = 0.5, size = 8, colour = "grey40",
                                     margin = margin(t = 8, b = 8)),
      legend.position = "none",
      plot.margin     = margin(t = 12, r = 2, b = 12, l = 2)
    )

  p_cort / p_subc + plot_layout(heights = c(3, 2))
}

# ── Figure 7c-1: Unthresholded ─────────────────────────────────────────────────
panels_unc <- lapply(names(contrasts), function(cn) make_dka_panel(cn, fdr_only = FALSE))

fig_unc <- wrap_plots(panels_unc, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "DesikanAseg Volume: Volume Reduction Across the Disease Continuum (Unthresholded)",
    subtitle = paste0(
      "Pink/red = parcels with reduced volume in the first-named group (|Hedges' g|, unthresholded).\n",
      "Grey = no volume reduction or greater volume relative to the reference group.\n",
      "Cortical views: LH lateral | LH medial | RH medial | RH lateral.  Subcortical: coronal slice.\n",
      "FDR correction applied jointly across 82 parcels (68 cortical + 14 subcortical)."
    ),
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(size = 14, face = "bold", hjust = 0.5),
      plot.subtitle = element_text(size = 10, colour = "grey40", hjust = 0.5)
    )
  ) & theme(legend.position = "right")

options(repr.plot.width = 10, repr.plot.height = 19)
print(fig_unc)
ggsave(file.path(FIG_DIR, "figure7c1_desikanaseg_volume_unthresholded.png"),
       fig_unc, width = 10, height = 19, dpi = 300)
cat("Saved: figure7c1_desikanaseg_volume_unthresholded.png\n")

# ── Figure 7c-2: FDR-corrected ─────────────────────────────────────────────────
panels_fdr <- lapply(names(contrasts), function(cn) make_dka_panel(cn, fdr_only = TRUE))

fig_fdr <- wrap_plots(panels_fdr, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "DesikanAseg Volume: FDR-Corrected Volume Reduction",
    subtitle = paste0(
      "Pink/red = parcels with FDR-significant volume reduction (q < 0.05, jointly across 82 parcels).\n",
      "Grey = not FDR-significant.  No parcel survives FDR correction in any contrast.\n",
      "Cortical views: LH lateral | LH medial | RH medial | RH lateral.  Subcortical: coronal slice."
    ),
    tag_levels = "a",
    theme = theme(
      plot.title    = element_text(size = 14, face = "bold", hjust = 0.5),
      plot.subtitle = element_text(size = 10, colour = "grey40", hjust = 0.5)
    )
  ) & theme(legend.position = "right")

options(repr.plot.width = 10, repr.plot.height = 19)
print(fig_fdr)
ggsave(file.path(FIG_DIR, "figure7c2_desikanaseg_volume_fdr_corrected.png"),
       fig_fdr, width = 10, height = 19, dpi = 300)
cat("Saved: figure7c2_desikanaseg_volume_fdr_corrected.png\n")

### 8. Results table

All parcels sorted by absolute Hedges' g. `*` = FDR significant (q < 0.05).

In [ ]:
df_stats %>%
  select(contrast, type, hemi, region, g, Tvalue, pvalue, p_fdr) %>%
  mutate(
    across(where(is.numeric), \(x) round(x, 4)),
    sig = if_else(p_fdr < 0.05, "*", "")
  ) %>%
  arrange(contrast, type, desc(abs(g)))

---
## Notes

### FDR correction scope
FDR is applied across all 82 parcels (68 cortical + 14 subcortical) within each contrast.
This is more conservative than correcting cortical and subcortical separately, so FDR-sig
thresholds here differ from those in `ggseg_volume_subcortical.ipynb` (14 tests only).

### Data source
Effect sizes derived from pre-computed OLS T-statistics:
- Cortical: `results/*_parcelwise_ttest_volume_cortical.csv` (from `volume_cortical_shi.ipynb`)
- Subcortical: `results/*_parcelwise_ttest_volume_subcortical.csv` (from `volume_subcortical_shi.ipynb`)

**eTIV correction**: All T-statistics include eTIV as an OLS covariate. This is critical for
subcortical volumes especially — without eTIV correction, PD groups (larger TIV in PPMI)
appear to have *larger* subcortical volumes, which reverses the true effect direction.

### ggseg 2.x API
- `dk()` and `aseg()` are functions (not data objects)
- Join key: single `label` column - only pass `label` + fill variable to `ggplot()`
- `position_brain("stacked")` crashes in v2.1.0; use `"horizontal"` for cortical
- aseg atlas does not need a `position` argument (default layout works)

**Group codes:** CONCOHORT 1 = De Novo PD · 2 = HC · 4 = Prodromal PD

### 6g. Diverging pastel maps: cortical + subcortical combined (light pink / light blue)

**Appendix figure** — same pastel diverging style as Figure 5e (cortical thickness) but applied to
the combined DesikanAseg volume analysis (68 cortical + 14 subcortical parcels):

- **Light pink** (`lightpink`): g < 0 — first-named group has *less* volume
- **Light blue** (`#AED6F1`): g > 0 — first-named group has *more* volume
- **White**: no difference; **Grey85**: no data

Top row per panel = cortical dk atlas (4 views). Bottom row = subcortical aseg (coronal slice).
All parcels shown regardless of significance. Scale limits ±0.30 cortical, ±0.50 subcortical.

In [ ]:
CNAMES_DISPLAY_DKA <- c(
  "De Novo PD vs HC"           = "de novo PD vs HC",
  "Prodromal PD vs HC"         = "prodromal PD vs HC",
  "De Novo PD vs Prodromal PD" = "de novo PD vs prodromal PD"
)

G_LIMIT_PASTEL <- 0.30

scale_pastel_dka <- scale_fill_gradient2(
  low = "lightpink", mid = "white", high = "#AED6F1",
  midpoint = 0, limits = c(-G_LIMIT_PASTEL, G_LIMIT_PASTEL),
  oob = scales::squish, name = "Hedges' g", na.value = "grey85"
)

if (!exists("dk_4view")) {
  dk_4view <- dk()
  dk_4view$data[[1]] <- dk_4view$data[[1]] %>% filter(view %in% c("lateral", "medial"))
}
if (!exists("coronal_join")) {
  coronal_join <- function(df) brain_join(df, aseg()) %>% dplyr::filter(view == "coronal_1")
}

make_pastel_dka_panel <- function(cname, fdr_only = FALSE) {
  d_cort <- df_stats %>% filter(contrast == cname, type == "cortical")
  d_subc <- df_stats %>% filter(contrast == cname, type == "subcortical")

  if (fdr_only) {
    d_cort <- d_cort %>% mutate(g_plot = if_else(p_fdr < 0.05, g, NA_real_))
    d_subc <- d_subc %>% mutate(g_plot = if_else(p_fdr < 0.05, g, NA_real_))
  } else {
    d_cort <- d_cort %>% mutate(g_plot = g)
    d_subc <- d_subc %>% mutate(g_plot = g)
  }

  p_cort <- ggplot(d_cort %>% select(label, g_plot)) +
    geom_brain(atlas = dk_4view, mapping = aes(fill = g_plot), colour = "grey70",
               position = position_brain("horizontal")) +
    scale_pastel_dka +
    labs(title = CNAMES_DISPLAY_DKA[[cname]]) +
    theme_void() +
    theme(plot.title    = element_text(hjust = 0.5, size = 11, face = "bold", margin = margin(t = 8, b = 3)),
          legend.position = "none", plot.margin = margin(2, 2, 2, 2))

  p_subc <- ggplot(coronal_join(d_subc %>% select(label, g_plot))) +
    geom_sf(aes(fill = g_plot), colour = "grey70") +
    scale_pastel_dka +
    labs() +
    theme_void() +
    theme(legend.position = "none", plot.margin = margin(8, 2, 8, 2))

  p_cort / p_subc + plot_layout(heights = c(3, 2))
}

# Figure 6g-1: unthresholded
options(repr.plot.width = 10, repr.plot.height = 19)

panels_unc <- lapply(names(contrasts), function(cn) make_pastel_dka_panel(cn, fdr_only = FALSE))

fig_dka_unc <- wrap_plots(panels_unc, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "DesikanAseg Volume: Diverging Hedges' g (unthresholded)",
    subtitle = paste0("Light pink = volume loss (g < 0) | Light blue = volume gain (g > 0)\n",
                      "Scale ±0.30 | All parcels shown"),
    tag_levels = "a",
    theme = theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
                  plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40"))
  ) & theme(legend.position = "right")

print(fig_dka_unc)
ggsave(file.path(FIG_DIR, "appendix_diverging_volume_desikanaseg_unthresh.png"),
       fig_dka_unc, width = 10, height = 19, dpi = 300)
cat("Saved: appendix_diverging_volume_desikanaseg_unthresh.png\n")

# Figure 6g-2: FDR-masked
panels_fdr <- lapply(names(contrasts), function(cn) make_pastel_dka_panel(cn, fdr_only = TRUE))

fig_dka_fdr <- wrap_plots(panels_fdr, ncol = 1) +
  plot_layout(guides = "collect") +
  plot_annotation(
    title    = "DesikanAseg Volume: Diverging Hedges' g (FDR q < 0.05)",
    subtitle = "Light pink / blue = FDR-significant parcels only | Grey = not significant",
    tag_levels = "a",
    theme = theme(plot.title    = element_text(hjust = 0.5, size = 13, face = "bold"),
                  plot.subtitle = element_text(hjust = 0.5, size = 9,  colour = "grey40"))
  ) & theme(legend.position = "right")

print(fig_dka_fdr)
ggsave(file.path(FIG_DIR, "appendix_diverging_volume_desikanaseg_fdr.png"),
       fig_dka_fdr, width = 10, height = 19, dpi = 300)
cat("Saved: appendix_diverging_volume_desikanaseg_fdr.png\n")